# 06 - Merge Sources

## Objetivo
Este notebook une las fuentes ya reconstruidas en `data/interim/` para producir un dataset integrado en `data/processed/`.

## Fuentes esperadas
- `milking.parquet`
- `rumination.parquet`
- `weather.parquet`
- `pdf_events.parquet`

## Criterio de integración
La clave principal del merge será:

- `cow_id`
- `date`

Para clima, normalmente la unión se hace solo por `date`.

## Alcance
Este notebook:
- carga las fuentes disponibles,
- normaliza claves,
- agrega por día cuando sea necesario,
- une todo en un dataset integrado,
- guarda `training_dataset.parquet`.

Este notebook **no entrena modelos**.

In [45]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Rutas del proyecto

In [46]:
CURRENT = Path.cwd().resolve()

if (CURRENT / "data" / "interim").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data" / "interim").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise FileNotFoundError("No se encontró data/interim ni en el directorio actual ni en el padre.")

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("INTERIM_DIR  :", INTERIM_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT : C:\Users\PC\Documents\GitHub\ProyectoGranja
INTERIM_DIR  : C:\Users\PC\Documents\GitHub\ProyectoGranja\data\interim
PROCESSED_DIR: C:\Users\PC\Documents\GitHub\ProyectoGranja\data\processed


## 2. Ubicación de archivos esperados

In [47]:
MILKING_PATH = INTERIM_DIR / "milking.parquet"
RUMINATION_PATH = INTERIM_DIR / "rumination.parquet"
WEATHER_PATH = INTERIM_DIR / "clima.parquet"
PDF_EVENTS_PATH = INTERIM_DIR / "pdf_events.parquet"

for p in [MILKING_PATH, RUMINATION_PATH, WEATHER_PATH, PDF_EVENTS_PATH]:
    print(f"{p.name:22} -> {'OK' if p.exists() else 'MISSING'}")

milking.parquet        -> OK
rumination.parquet     -> OK
clima.parquet          -> OK
pdf_events.parquet     -> OK


## 3. Funciones auxiliares

In [48]:
def LoadParquetIfExists(path: Path):
    if path.exists():
        df = pd.read_parquet(path)
        print(f"Loaded {path.name:22} -> {df.shape}")
        return df
    print(f"Skipped {path.name:21} -> file not found")
    return None


def EnsureDateColumn(df: pd.DataFrame, datetime_candidates, out_col="date"):
    df = df.copy()

    if out_col in df.columns:
        df[out_col] = pd.to_datetime(df[out_col], errors="coerce").dt.normalize()
        return df

    for col in datetime_candidates:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            df[out_col] = df[col].dt.normalize()
            return df

    raise KeyError(f"No se pudo crear '{out_col}'. Columnas candidatas no encontradas: {datetime_candidates}")


def NormalizeCowId(df: pd.DataFrame):
    df = df.copy()

    candidate_cols = ["cow_id", "animal_id", "vid", "resolved_vid", "resolved_cow_id"]
    for col in candidate_cols:
        if col in df.columns:
            df["cow_id"] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
            return df

    return df


def DailyAggMilking(df: pd.DataFrame):
    df = NormalizeCowId(df)
    df = EnsureDateColumn(df, ["hora_inicio", "timestamp", "datetime", "date"])

    numeric_candidates = [
        "produccion_kg", "di", "dd", "ti", "td",
        "duracion_min", "intervalo_ordeno_min",
        "patada", "incompleto", "pezones_no_encontrados"
    ]

    existing_num = [c for c in numeric_candidates if c in df.columns]
    for c in existing_num:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    agg = {}
    if "produccion_kg" in df.columns: agg["produccion_kg"] = "sum"
    if "di" in df.columns: agg["di"] = "sum"
    if "dd" in df.columns: agg["dd"] = "sum"
    if "ti" in df.columns: agg["ti"] = "sum"
    if "td" in df.columns: agg["td"] = "sum"
    if "numero_ordeno" in df.columns: agg["numero_ordeno"] = "count"
    if "duracion_min" in df.columns: agg["duracion_min"] = "sum"
    if "intervalo_ordeno_min" in df.columns: agg["intervalo_ordeno_min"] = "mean"
    if "patada" in df.columns: agg["patada"] = "sum"
    if "incompleto" in df.columns: agg["incompleto"] = "sum"
    if "pezones_no_encontrados" in df.columns: agg["pezones_no_encontrados"] = "sum"

    cat_keep = {}
    for c in ["ubre", "destino_leche", "ms"]:
        if c in df.columns:
            cat_keep[c] = "first"

    daily = (
        df.groupby(["cow_id", "date"], dropna=False)
          .agg({**agg, **cat_keep})
          .reset_index()
    )

    rename_map = {
        "numero_ordeno": "ordeños_dia",
        "duracion_min": "duracion_total_min",
        "intervalo_ordeno_min": "intervalo_ordeno_prom_min",
        "patada": "patadas_dia",
        "incompleto": "incompletos_dia",
        "pezones_no_encontrados": "pezones_no_encontrados_dia"
    }
    daily = daily.rename(columns=rename_map)

    return daily


def DailyAggRumination(df: pd.DataFrame):
    df = NormalizeCowId(df)
    df = EnsureDateColumn(df, ["date", "timestamp", "datetime", "fecha"])

    rename_map = {
        "ruminating_minutes": "rumia_min",
        "ruminating": "rumia_min",
        "resolved_group": "group_id",
        "group_id": "group_id",
        "days_in_milk": "days_in_milk",
        "lactation_age": "lactation_age",
        "weekday": "weekday",
        "month": "month",
    }
    for old, new in rename_map.items():
        if old in df.columns and old != new:
            df = df.rename(columns={old: new})

    agg = {}
    for c in ["rumia_min", "days_in_milk", "lactation_age", "weekday", "month"]:
        if c in df.columns:
            agg[c] = "mean"
    for c in ["group_id", "source_file"]:
        if c in df.columns:
            agg[c] = "first"

    daily = (
        df.groupby(["cow_id", "date"], dropna=False)
          .agg(agg)
          .reset_index()
    )
    return daily


def DailyAggWeather(df: pd.DataFrame):
    df = df.copy()

    # La fuente de clima usa "time"
    df = EnsureDateColumn(df, ["time", "date", "datetime", "timestamp", "fecha"])

    rename_map = {
        "temperatura": "temperature_c",
        "temperature": "temperature_c",
        "temperature_2m": "temperature_c",
        "humedad": "humidity_pct",
        "humidity": "humidity_pct",
        "relative_humidity_2m": "humidity_pct",
        "lluvia": "rain_mm",
        "rain": "rain_mm",
        "precipitation": "rain_mm",
        "viento": "wind_speed",
        "wind": "wind_speed",
        "wind_speed_10m": "wind_speed",
        "pressure_msl": "pressure_msl",
    }

    for old, new in rename_map.items():
        if old in df.columns and old != new:
            df = df.rename(columns={old: new})

    numeric_cols = [
        c for c in [
            "temperature_c",
            "humidity_pct",
            "pressure_msl",
            "rain_mm",
            "wind_speed",
            "heat_index",
        ]
        if c in df.columns
    ]

    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if numeric_cols:
        daily = df.groupby("date", dropna=False)[numeric_cols].mean().reset_index()
    else:
        daily = df[["date"]].drop_duplicates().copy()

    return daily


def DailyAggPdfEvents(df: pd.DataFrame):
    df = NormalizeCowId(df)
    df = EnsureDateColumn(df, ["event_date", "date", "fecha_evento", "timestamp", "datetime"])

    if "raw_event_text" not in df.columns:
        text_col = None
        for c in ["evento", "descripcion", "event_text", "raw_text"]:
            if c in df.columns:
                text_col = c
                break
        if text_col:
            df = df.rename(columns={text_col: "raw_event_text"})
        else:
            df["raw_event_text"] = ""

    df["raw_event_text"] = df["raw_event_text"].astype(str)

    daily = (
        df.groupby(["cow_id", "date"], dropna=False)
          .agg(
              eventos_pdf_count=("raw_event_text", "count"),
              eventos_pdf_text=("raw_event_text", lambda s: " | ".join(s.dropna().astype(str).head(10)))
          )
          .reset_index()
    )

    return daily

## 4. Cargar fuentes

In [49]:
milking_raw = LoadParquetIfExists(MILKING_PATH)
rumination_raw = LoadParquetIfExists(RUMINATION_PATH)
weather_raw = LoadParquetIfExists(WEATHER_PATH)
pdf_events_raw = LoadParquetIfExists(PDF_EVENTS_PATH)

Loaded milking.parquet        -> (23763, 17)
Loaded rumination.parquet     -> (19110, 15)
Loaded clima.parquet          -> (5880, 6)
Loaded pdf_events.parquet     -> (6890, 6)


## 5. Agregación diaria por fuente

In [50]:
milking_daily = DailyAggMilking(milking_raw) if milking_raw is not None else None
rumination_daily = DailyAggRumination(rumination_raw) if rumination_raw is not None else None
weather_daily = DailyAggWeather(weather_raw) if weather_raw is not None else None
pdf_events_daily = DailyAggPdfEvents(pdf_events_raw) if pdf_events_raw is not None else None

for name, df_src in [
    ("milking_daily", milking_daily),
    ("rumination_daily", rumination_daily),
    ("weather_daily", weather_daily),
    ("pdf_events_daily", pdf_events_daily),
]:
    if df_src is None:
        print(f"{name:18} -> None")
    else:
        print(f"{name:18} -> {df_src.shape}")

milking_daily      -> (9814, 13)
rumination_daily   -> (7098, 10)
weather_daily      -> (245, 6)
pdf_events_daily   -> (561, 4)


## 6. Vista rápida de cada fuente agregada

In [51]:
if milking_daily is not None:
    display(milking_daily.head())

if rumination_daily is not None:
    display(rumination_daily.head())

if weather_daily is not None:
    display(weather_daily.head())

if pdf_events_daily is not None:
    display(pdf_events_daily.head())

,cow_id,date,produccion_kg,di,dd,ti,td,ordeños_dia,duracion_total_min,intervalo_ordeno_prom_min,ubre,destino_leche,ms
0,1204,2025-01-01,16.68,5.16,4.64,0.00,6.88,1,0.0,NaN,0,Tanque,VMS 1
1,1204,2025-01-02,17.91,2.65,4.44,5.75,5.07,2,0.0,NaN,1,Divert 3,VMS 1
2,1204,2025-01-03,25.36,6.26,6.52,2.77,9.81,2,0.0,NaN,1,Tanque,VMS 1
3,1204,2025-01-04,16.71,4.13,3.89,2.96,5.73,1,0.0,NaN,1,Tanque,VMS 1
4,1204,2025-01-05,25.04,5.85,5.94,4.85,8.40,2,0.0,NaN,1,Tanque,VMS 1


,cow_id,date,rumia_min,days_in_milk,lactation_age,weekday,month,group_id,group_id,source_file
0,1204,2025-06-27,NaN,417.0,417.0,4.0,6.0,100.0,100.0,group_100_ruminating_rumia.csv
1,1204,2025-06-28,657.0,418.0,418.0,5.0,6.0,100.0,100.0,group_100_ruminating_rumia.csv
2,1204,2025-06-29,495.0,419.0,419.0,6.0,6.0,100.0,100.0,group_100_ruminating_rumia.csv
3,1204,2025-06-30,549.0,420.0,420.0,0.0,6.0,100.0,100.0,group_100_ruminating_rumia.csv
4,1204,2025-07-01,566.0,421.0,421.0,1.0,7.0,100.0,100.0,group_100_ruminating_rumia.csv


,date,temperature_c,humidity_pct,pressure_msl,rain_mm,wind_speed
0,2025-01-01,14.266667,43.291667,1019.283333,0.000000,11.150000
1,2025-01-02,13.404167,64.916667,1022.504167,0.000000,13.187500
2,2025-01-03,12.791667,73.000000,1023.987500,0.000000,16.045833
3,2025-01-04,14.333333,70.750000,1020.816667,0.029167,7.800000
4,2025-01-05,14.333333,58.458333,1018.504167,0.000000,8.445833


,cow_id,date,eventos_pdf_count,eventos_pdf_text
0,1204,2023-07-27,1,"Invitación Visita 13/07/2 407), Available for ..."
1,1204,2025-05-29,1,Invitación Visita 05/06/2 Gestacion (>40 dias ...
2,1204,2025-06-05,1,Invitación Visita 19/06/2 Gestacion (>40 dias ...
3,1204,2025-06-19,1,"Control de Gest 19/06/2 User1 19/06/2025, Diag..."
4,1204,2025-07-24,1,"Control de Gest 24/07/2 User1 24/07/2025, Reco..."


## 7. Seleccionar base del merge

La base recomendada es `milking_daily` cuando exista, porque suele contener la producción diaria objetivo.
Si no existe, se usa `rumination_daily`.

In [52]:
if milking_daily is not None:
    merged = milking_daily.copy()
    print("Base del merge: milking_daily")
elif rumination_daily is not None:
    merged = rumination_daily.copy()
    print("Base del merge: rumination_daily")
else:
    raise ValueError("No existe una base válida. Se requiere al menos milking.parquet o rumination.parquet.")

Base del merge: milking_daily


## 8. Merge con rumia

In [53]:
if rumination_daily is not None and "rumia_min" in rumination_daily.columns:
    if not (merged is rumination_daily):
        merged = merged.merge(
            rumination_daily,
            on=["cow_id", "date"],
            how="left",
            suffixes=("", "_rum")
        )

print(merged.shape)
merged.head()

(9814, 21)


,cow_id,date,produccion_kg,di,dd,ti,td,ordeños_dia,duracion_total_min,intervalo_ordeno_prom_min,ubre,destino_leche,ms,rumia_min,days_in_milk,lactation_age,weekday,month,group_id,group_id,source_file
0,1204,2025-01-01,16.68,5.16,4.64,0.00,6.88,1,0.0,NaN,0,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1204,2025-01-02,17.91,2.65,4.44,5.75,5.07,2,0.0,NaN,1,Divert 3,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1204,2025-01-03,25.36,6.26,6.52,2.77,9.81,2,0.0,NaN,1,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1204,2025-01-04,16.71,4.13,3.89,2.96,5.73,1,0.0,NaN,1,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1204,2025-01-05,25.04,5.85,5.94,4.85,8.40,2,0.0,NaN,1,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 9. Merge con clima

In [54]:
if weather_daily is not None:
    merged = merged.merge(
        weather_daily,
        on="date",
        how="left",
        suffixes=("", "_weather")
    )

print(merged.shape)
merged.head()

(9814, 26)


,cow_id,date,produccion_kg,di,dd,ti,td,ordeños_dia,duracion_total_min,intervalo_ordeno_prom_min,ubre,destino_leche,ms,rumia_min,days_in_milk,lactation_age,weekday,month,group_id,group_id,source_file,temperature_c,humidity_pct,pressure_msl,rain_mm,wind_speed
0,1204,2025-01-01,16.68,5.16,4.64,0.00,6.88,1,0.0,NaN,0,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.266667,43.291667,1019.283333,0.000000,11.150000
1,1204,2025-01-02,17.91,2.65,4.44,5.75,5.07,2,0.0,NaN,1,Divert 3,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.404167,64.916667,1022.504167,0.000000,13.187500
2,1204,2025-01-03,25.36,6.26,6.52,2.77,9.81,2,0.0,NaN,1,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.791667,73.000000,1023.987500,0.000000,16.045833
3,1204,2025-01-04,16.71,4.13,3.89,2.96,5.73,1,0.0,NaN,1,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.333333,70.750000,1020.816667,0.029167,7.800000
4,1204,2025-01-05,25.04,5.85,5.94,4.85,8.40,2,0.0,NaN,1,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.333333,58.458333,1018.504167,0.000000,8.445833


## 10. Merge con eventos PDF

In [55]:
if pdf_events_daily is not None:
    merged = merged.merge(
        pdf_events_daily,
        on=["cow_id", "date"],
        how="left",
        suffixes=("", "_pdf")
    )

print(merged.shape)
merged.head()

(9814, 28)


,cow_id,date,produccion_kg,di,dd,ti,td,ordeños_dia,duracion_total_min,intervalo_ordeno_prom_min,ubre,destino_leche,ms,rumia_min,days_in_milk,lactation_age,weekday,month,group_id,group_id,source_file,temperature_c,humidity_pct,pressure_msl,rain_mm,wind_speed,eventos_pdf_count,eventos_pdf_text
0,1204,2025-01-01,16.68,5.16,4.64,0.00,6.88,1,0.0,NaN,0,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.266667,43.291667,1019.283333,0.000000,11.150000,NaN,NaN
1,1204,2025-01-02,17.91,2.65,4.44,5.75,5.07,2,0.0,NaN,1,Divert 3,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.404167,64.916667,1022.504167,0.000000,13.187500,NaN,NaN
2,1204,2025-01-03,25.36,6.26,6.52,2.77,9.81,2,0.0,NaN,1,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.791667,73.000000,1023.987500,0.000000,16.045833,NaN,NaN
3,1204,2025-01-04,16.71,4.13,3.89,2.96,5.73,1,0.0,NaN,1,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.333333,70.750000,1020.816667,0.029167,7.800000,NaN,NaN
4,1204,2025-01-05,25.04,5.85,5.94,4.85,8.40,2,0.0,NaN,1,Tanque,VMS 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.333333,58.458333,1018.504167,0.000000,8.445833,NaN,NaN


## 11. Variables temporales derivadas mínimas

Estas variables son útiles y seguras para el dataset integrado:
- año
- mes
- día
- día de la semana

In [56]:
merged["date"] = pd.to_datetime(merged["date"], errors="coerce")
merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day
merged["weekday"] = merged["date"].dt.weekday

## 12. Orden y revisión de nulos

In [57]:
sort_cols = [c for c in ["cow_id", "date"] if c in merged.columns]
if sort_cols:
    merged = merged.sort_values(sort_cols).reset_index(drop=True)

nan_report = pd.DataFrame({
    "column": merged.columns,
    "nan_count": merged.isna().sum().values,
    "nan_pct": (merged.isna().mean() * 100).values
}).sort_values("nan_pct", ascending=False)

print("Shape final:", merged.shape)
display(nan_report.head(30))

Shape final: (9814, 30)


,column,nan_count,nan_pct
9,intervalo_ordeno_prom_min,9814,100.000000
13,rumia_min,9787,99.724883
18,group_id,9786,99.714693
19,group_id,9786,99.714693
15,lactation_age,9786,99.714693
14,days_in_milk,9786,99.714693
20,source_file,9786,99.714693
27,eventos_pdf_text,9666,98.491950
26,eventos_pdf_count,9666,98.491950
21,temperature_c,35,0.356633


## 13. Validaciones rápidas

In [58]:
print("Rango de fechas:")
print(merged["date"].min(), "->", merged["date"].max())

if "cow_id" in merged.columns:
    print("\nNúmero de vacas:", merged["cow_id"].nunique(dropna=True))

if "produccion_kg" in merged.columns:
    print("\nProducción total registrada:", merged["produccion_kg"].sum())

if "rumia_min" in merged.columns:
    print("Rumia total registrada:", merged["rumia_min"].sum())

Rango de fechas:
2025-01-01 00:00:00 -> 2025-09-30 00:00:00

Número de vacas: 65

Producción total registrada: 368634.65
Rumia total registrada: 7036.0


## 14. Guardar dataset integrado

In [59]:
OUTPUT_PATH = PROCESSED_DIR / "training_dataset.parquet"
merged.to_parquet(OUTPUT_PATH, index=False)

print("Dataset guardado en:")
print(OUTPUT_PATH)

ValueError: Duplicate column names found: ['cow_id', 'date', 'produccion_kg', 'di', 'dd', 'ti', 'td', 'ordeños_dia', 'duracion_total_min', 'intervalo_ordeno_prom_min', 'ubre', 'destino_leche', 'ms', 'rumia_min', 'days_in_milk', 'lactation_age', 'weekday', 'month', 'group_id', 'group_id', 'source_file', 'temperature_c', 'humidity_pct', 'pressure_msl', 'rain_mm', 'wind_speed', 'eventos_pdf_count', 'eventos_pdf_text', 'year', 'day']

## 15. Export opcional a CSV

In [ ]:
CSV_OUTPUT_PATH = PROCESSED_DIR / "training_dataset.csv"
merged.to_csv(CSV_OUTPUT_PATH, index=False)

print("CSV guardado en:")
print(CSV_OUTPUT_PATH)

## 16. Próximo paso

Con este archivo ya puedes pasar a:
- `07_feature_engineering.ipynb`
- `08_model_training.ipynb`

Si luego agregas nuevas fuentes, este notebook debe seguir siendo el punto central de integración.

In [ ]:
print("Milking range   :", milking_daily["date"].min(), "->", milking_daily["date"].max())
print("Rumination range:", rumination_daily["date"].min(), "->", rumination_daily["date"].max())
print("PDF range       :", pdf_events_daily["date"].min(), "->", pdf_events_daily["date"].max())

Milking range   : 2025-01-01 00:00:00 -> 2025-09-30 00:00:00
Rumination range: 2025-06-27 00:00:00 -> 2025-09-25 00:00:00
PDF range       : 2022-05-26 00:00:00 -> 2025-09-25 00:00:00


In [ ]:
milking_keys = set(zip(milking_daily["cow_id"], milking_daily["date"]))

r0 = rumination_daily.copy()
r1 = rumination_daily.copy()
r_1 = rumination_daily.copy()

r1["date"] = r1["date"] + pd.Timedelta(days=1)
r_1["date"] = r_1["date"] - pd.Timedelta(days=1)

print("Exact match :", len(milking_keys & set(zip(r0["cow_id"], r0["date"]))))
print("+1 day      :", len(milking_keys & set(zip(r1["cow_id"], r1["date"]))))
print("-1 day      :", len(milking_keys & set(zip(r_1["cow_id"], r_1["date"]))))

Exact match : 28
+1 day      : 30
-1 day      : 27
